## Motorway Network — Official LGL WFS

### Motorway Network — Data Source and Discovery

**Objective:** Obtain official motorway geometries for Baden-Württemberg to support spatial constraints and roadside solar PV eligibility analysis.

**Data source:** LGL Baden-Württemberg — INSPIRE Transport Networks (ATKIS Basis-DLM), accessed via a Web Feature Service (WFS).

**Data discovery:**
- Identified the official transport network through Geoportal BW.
- Inspected WFS `GetCapabilities` and `DescribeFeatureType` XML to understand available features and attributes.
- Identified `FormOfWay` as the road classification feature and `RoadLink` as the corresponding line geometry.
- Verified that an OGC XML filter can retrieve `FormOfWay = motorway` records, each referencing a `RoadLink` identifier.

**Approach:** Retrieve motorway classifications → extract referenced RoadLink IDs → obtain line geometries → construct a GeoDataFrame (EPSG:25832).

**Methodological note:** Motorway centrelines will be used to derive spatial buffers. Physical road exclusions and the 500 m EEG roadside eligibility corridor will be treated separately.

**Source:** [LGL INSPIRE Transport Network WFS](https://owsproxy.lgl-bw.de/owsproxy/wfs/WFS_INSP_BW_Verkehrsnetz_ATKIS_BasisDLM?SERVICE=WFS&REQUEST=GetCapabilities)

In [1]:
import requests
import xml.etree.ElementTree as ET
import geopandas as gpd
import matplotlib.pyplot as plt

import time


from shapely.geometry import LineString
from tqdm.auto import tqdm


In [2]:
WFS_URL = (
    "https://owsproxy.lgl-bw.de/owsproxy/wfs/"
    "WFS_INSP_BW_Verkehrsnetz_ATKIS_BasisDLM"
)

motorway_filter = """
<fes:Filter
    xmlns:fes="http://www.opengis.net/fes/2.0"
    xmlns:tn-ro="http://inspire.ec.europa.eu/schemas/tn-ro/4.0"
    xmlns:xlink="http://www.w3.org/1999/xlink">
    <fes:PropertyIsEqualTo>
        <fes:ValueReference>tn-ro:formOfWay/@xlink:href</fes:ValueReference>
        <fes:Literal>https://inspire.ec.europa.eu/codelist/FormOfWayValue/motorway</fes:Literal>
    </fes:PropertyIsEqualTo>
</fes:Filter>
"""

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "tn-ro:FormOfWay",
    "COUNT": 5,
    "FILTER": motorway_filter,
}

response = requests.get(WFS_URL, params=params, timeout=60)
response.raise_for_status()

root = ET.fromstring(response.content)

print("HTTP status:", response.status_code)
print("Response root:", root.tag)
print("Response size:", len(response.content), "bytes")

HTTP status: 200
Response root: {http://www.opengis.net/wfs/2.0}FeatureCollection
Response size: 10564 bytes


In [3]:
namespaces = {
    "wfs": "http://www.opengis.net/wfs/2.0",
    "tn-ro": "http://inspire.ec.europa.eu/schemas/tn-ro/4.0",
    "net": "http://inspire.ec.europa.eu/schemas/net/4.0",
    "xlink": "http://www.w3.org/1999/xlink",
}

roadlink_ids = []

for feature in root.findall(".//tn-ro:FormOfWay", namespaces):
    road_class = feature.find("tn-ro:formOfWay", namespaces)
    road_ref = feature.find(".//net:element", namespaces)

    if road_class is None or road_ref is None:
        continue

    classification = road_class.get(
        f"{{{namespaces['xlink']}}}href"
    )

    reference = road_ref.get(
        f"{{{namespaces['xlink']}}}href"
    )

    if classification and classification.endswith("/motorway") and reference:
        roadlink_id = reference.split("#")[-1]
        roadlink_ids.append(roadlink_id)

print("Motorway RoadLink IDs:", len(roadlink_ids))
print(roadlink_ids)

Motorway RoadLink IDs: 5
['RoadLink.DEBWB0010000SC0e', 'RoadLink.DEBWB0010000SC3F', 'RoadLink.DEBWB00100010iRi', 'RoadLink.DEBWB0010001atat', 'RoadLink.DEBWB0010000SC4c']


In [4]:
namespaces["gml"] = "http://www.opengis.net/gml/3.2"

motorway_features = []

for roadlink_id in roadlink_ids:

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "STOREDQUERY_ID": "urn:ogc:def:query:OGC-WFS::GetFeatureById",
        "ID": roadlink_id,
    }

    response = requests.get(WFS_URL, params=params, timeout=60)
    response.raise_for_status()

    road_root = ET.fromstring(response.content)

    pos_list = road_root.find(
        ".//gml:posList",
        namespaces
    )

    if pos_list is None:
        print(f"No geometry found: {roadlink_id}")
        continue

    coordinates = [
        float(value)
        for value in pos_list.text.split()
    ]

    points = list(zip(coordinates[::2], coordinates[1::2]))

    motorway_features.append({
        "roadlink_id": roadlink_id,
        "geometry": LineString(points)
    })

print("Geometries retrieved:", len(motorway_features))

Geometries retrieved: 5


In [5]:
motorways_gdf = gpd.GeoDataFrame(
    motorway_features,
    geometry="geometry",
    crs="EPSG:25832"
)

print(motorways_gdf.head())
print("CRS:", motorways_gdf.crs)
print("Geometry types:", motorways_gdf.geom_type.unique())

                 roadlink_id  \
0  RoadLink.DEBWB0010000SC0e   
1  RoadLink.DEBWB0010000SC3F   
2  RoadLink.DEBWB00100010iRi   
3  RoadLink.DEBWB0010001atat   
4  RoadLink.DEBWB0010000SC4c   

                                            geometry  
0  LINESTRING (393926.746 5304970.423, 393844.329...  
1  LINESTRING (393580.624 5304390.574, 393651.623...  
2  LINESTRING (393112.346 5303494.993, 393120.988...  
3  LINESTRING (393231.618 5303765.706, 393277.696...  
4  LINESTRING (390866.254 5278083.282, 390838.585...  
CRS: EPSG:25832
Geometry types: <StringArray>
['LineString']
Length: 1, dtype: str


In [6]:
count_params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "tn-ro:FormOfWay",
    "RESULTTYPE": "hits",
    "FILTER": motorway_filter,
}

response = requests.get(
    WFS_URL,
    params=count_params,
    timeout=60
)

response.raise_for_status()

count_root = ET.fromstring(response.content)

print("Root:", count_root.tag)
print("numberMatched:", count_root.get("numberMatched"))
print("numberReturned:", count_root.get("numberReturned"))
print("Response:", response.text[:1000])

Root: {http://www.opengis.net/wfs/2.0}FeatureCollection
numberMatched: 9175
numberReturned: 0
Response: <?xml version='1.0' encoding='UTF-8'?>
<wfs:FeatureCollection xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd" xmlns:wfs="http://www.opengis.net/wfs/2.0" timeStamp="2026-10-10T19:34:42Z" numberMatched="9175" numberReturned="0">
</wfs:FeatureCollection>


In [7]:
total_features = 9175
batch_size = 500

all_roadlink_ids = []
all_formofway_ids = []

for start_index in tqdm(range(0, total_features, batch_size)):

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "tn-ro:FormOfWay",
        "FILTER": motorway_filter,
        "STARTINDEX": start_index,
        "COUNT": batch_size,
    }

    response = requests.get(WFS_URL, params=params, timeout=120)
    response.raise_for_status()

    page_root = ET.fromstring(response.content)

    features = page_root.findall(
        ".//tn-ro:FormOfWay", namespaces
    )

    print(f"Page {start_index}: {len(features)} features")

    for feature in features:
        formofway_id = feature.get(
            "{http://www.opengis.net/gml/3.2}id"
        )

        road_ref = feature.find(
            ".//net:element", namespaces
        )

        if road_ref is None:
            continue

        reference = road_ref.get(
            f"{{{namespaces['xlink']}}}href"
        )

        if reference:
            roadlink_id = reference.split("#")[-1]

            all_formofway_ids.append(formofway_id)
            all_roadlink_ids.append(roadlink_id)

  0%|          | 0/19 [00:00<?, ?it/s]

Page 0: 500 features
Page 500: 500 features
Page 1000: 500 features
Page 1500: 500 features
Page 2000: 500 features
Page 2500: 500 features
Page 3000: 500 features
Page 3500: 500 features
Page 4000: 500 features
Page 4500: 500 features
Page 5000: 500 features
Page 5500: 500 features
Page 6000: 500 features
Page 6500: 500 features
Page 7000: 500 features
Page 7500: 500 features
Page 8000: 500 features
Page 8500: 500 features
Page 9000: 175 features


In [9]:
print("Expected features:", total_features)
print("Retrieved classifications:", len(all_formofway_ids))
print("Unique classifications:", len(set(all_formofway_ids)))
print("Unique RoadLink IDs:", len(set(all_roadlink_ids)))

print(
    "Duplicate classifications:",
    len(all_formofway_ids) - len(set(all_formofway_ids))
)

Expected features: 9175
Retrieved classifications: 9175
Unique classifications: 9175
Unique RoadLink IDs: 9175
Duplicate classifications: 0


In [8]:
batch_size = 100
motorway_records = []
failed_batches = []

unique_ids = list(dict.fromkeys(all_roadlink_ids))

for start in tqdm(range(0, len(unique_ids), batch_size)):

    batch_ids = unique_ids[start:start + batch_size]

    resource_filter = (
        '<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">'
        + "".join(
            f'<fes:ResourceId rid="{road_id}"/>'
            for road_id in batch_ids
        )
        + '</fes:Filter>'
    )

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "tn-ro:RoadLink",
        "FILTER": resource_filter,
    }

    try:
        response = requests.get(WFS_URL, params=params, timeout=120)
        response.raise_for_status()

        root = ET.fromstring(response.content)

        roadlinks = root.findall(".//tn-ro:RoadLink", namespaces)

        returned_ids = set()

        for road in roadlinks:
            road_id = road.get("{http://www.opengis.net/gml/3.2}id")

            pos_list = road.find(".//gml:posList", namespaces)

            if road_id is None or pos_list is None or not pos_list.text:
                continue

            coords = [float(x) for x in pos_list.text.split()]
            points = list(zip(coords[::2], coords[1::2]))

            if len(points) < 2:
                continue

            motorway_records.append({
                "roadlink_id": road_id,
                "geometry": LineString(points)
            })

            returned_ids.add(road_id)

        missing_ids = set(batch_ids) - returned_ids

        if missing_ids:
            failed_batches.append({
                "start": start,
                "missing_ids": sorted(missing_ids)
            })

    except (requests.RequestException, ET.ParseError, ValueError) as error:
        print(f"Batch {start} failed: {error}")

        failed_batches.append({
            "start": start,
            "missing_ids": batch_ids
        })

    time.sleep(0.2)

print("Geometry records retrieved:", len(motorway_records))
print("Batches with missing features:", len(failed_batches))

  0%|          | 0/92 [00:00<?, ?it/s]

Geometry records retrieved: 9175
Batches with missing features: 0


In [10]:
motorways_full = gpd.GeoDataFrame(
    motorway_records,
    geometry="geometry",
    crs="EPSG:25832"
)

print("Expected RoadLinks:", len(unique_ids))
print("Retrieved RoadLinks:", len(motorways_full))
print("Unique RoadLinks:", motorways_full["roadlink_id"].nunique())
print("Missing RoadLinks:",
      len(set(unique_ids) - set(motorways_full["roadlink_id"])))

print("Geometry types:")
print(motorways_full.geom_type.value_counts())

print("Invalid geometries:", (~motorways_full.is_valid).sum())
print("Empty geometries:", motorways_full.is_empty.sum())

Expected RoadLinks: 9175
Retrieved RoadLinks: 9175
Unique RoadLinks: 9175
Missing RoadLinks: 0
Geometry types:
LineString    9175
Name: count, dtype: int64
Invalid geometries: 0
Empty geometries: 0


In [13]:
from pathlib import Path

output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

motorway_path = output_dir / "bw_motorways.gpkg"

motorways_full.to_file(
    motorway_path,
    layer="motorways",
    driver="GPKG"
)

print("Saved:", motorway_path)

Saved: ..\..\data\processed\bw_motorways.gpkg
